# Tahap 2: Contextual Biasing melalui Hotwords
Subbab 3.4, 3.6.6, dan 3.6.8. Varian terpilih Tahap 1 dijalankan ulang atas data uji dengan daftar istilah tiap mata kuliah pada parameter `hotwords`. Hasilnya dibandingkan dengan baseline T1-E untuk rumusan masalah kedua.

```
[1] Daftar hotwords per mata kuliah  -> hotwords.csv, hotwords.json
[2] Transkripsi T2                   -> mentah/<ID>.json, waktu_transkripsi.jsonl
[3] Metrik dan log per data          -> metrik_blok.csv, metrik_t2.csv, rtf_ulangan.csv, verifikasi_jiwer.csv, log/
[4] Analisis terhadap baseline       -> statistik_deskriptif.csv, ambang.csv, wilcoxon.csv, perubahan_rekaman.csv,
                                        per_jalur.csv, perubahan_istilah.csv, kesalahan_istilah.csv, kandidat_halusinasi.csv
```

Parameter decoding, jumlah ulangan, dan percobaan awal diambil dari `run_meta_tahap1.json`, sehingga satu-satunya perbedaan dengan T1-E adalah `hotwords`. Bagian 2 mengukur waktu, jadi jalankan di perangkat Tabel 3.2 dengan kondisi yang sama seperti Tahap 1:
- laptop tersambung ke pengisi daya dengan mode daya kinerja terbaik;
- tampilan layar dialihkan ke Radeon 780M;
- aplikasi lain ditutup.

Bagian 2 dapat dihentikan lalu dilanjutkan; pasangan ulangan dan rekaman yang sudah selesai tidak diulang.

## 0. Persiapan

In [4]:
from pathlib import Path
import sys, os, json, math, datetime, importlib, subprocess, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import soundfile as sf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
import src.metrik as mt
import src.log_data as ld
import src.analisis as an
import src.transkripsi as tr
for m in (pt, mt, ld, an, tr):
    importlib.reload(m)

OUT        = ROOT / "data" / "output"
MANIFEST   = OUT / "manifest_rekaman.csv"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
FREKUENSI  = ROOT / "data" / "processed" / "glosarium" / "frekuensi_materi_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
ACUAN_DIR  = OUT / "praproses_teks" / "acuan"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
T1         = OUT / "tahap1"
T2         = OUT / "tahap2"
MENTAH     = T2 / "mentah"
KERJA      = T2 / "_kerja"
LOG_WAKTU  = T2 / "waktu_transkripsi.jsonl"
for d in (T2, MENTAH, KERJA):
    d.mkdir(parents=True, exist_ok=True)

rel = lambda p: Path(p).relative_to(ROOT).as_posix()

In [5]:
# Varian terpilih dan konfigurasi Tahap 1
pemilihan = json.loads((T1 / "pemilihan_varian.json").read_text(encoding="utf-8"))
meta_t1 = json.loads((T1 / "run_meta_tahap1.json").read_text(encoding="utf-8"))
VARIAN = pemilihan["terpilih"]
KODE_DASAR = pemilihan["kode_baseline"]
KODE = "T2"

manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False, encoding="utf-8-sig").set_index("file_id")
uji = sorted(manifest.index[manifest.split == "uji"])
assert len(uji) == 9, f"himpunan uji berisi {len(uji)} rekaman"

kamus = pt.muat_kamus(KAMUS_PATH)
assert kamus.terkunci and kamus.tanggal_kunci == meta_t1["kamus"]["tanggal_kunci"], "kamus berbeda dengan Tahap 1"
glosarium = pt.muat_glosarium(GLOSARIUM)
kanonik = (pd.read_csv(GLOSARIUM, dtype=str, keep_default_na=False, encoding="utf-8-sig")
             .drop_duplicates("entri_id").set_index("entri_id").bentuk_kanonik.to_dict())
acuan = {f: json.loads((ACUAN_DIR / f"{f}.json").read_text(encoding="utf-8")) for f in uji}
MK = {f: acuan[f]["mata_kuliah"] for f in uji}
norm = {f: pt.Normalisasi(kamus, glosarium[MK[f]]) for f in uji}

# Durasi dari WAV bila ada, lalu dicocokkan dengan durasi yang tercatat di Tahap 1
waktu_t1 = pd.DataFrame([json.loads(b) for b in (T1 / "waktu_transkripsi.jsonl").read_text(encoding="utf-8").splitlines()
                         if b.strip()])
durasi_t1 = waktu_t1.groupby("file_id").durasi_s.first()
audio_ada = {f: (AUDIO_PROC / f"{f}.wav").exists() for f in uji}
durasi = {}
for f in uji:
    if audio_ada[f]:
        i = sf.info(str(AUDIO_PROC / f"{f}.wav"))
        assert (i.samplerate, i.channels) == (16000, 1), f"{f}: {i.samplerate} Hz, {i.channels} kanal"
        durasi[f] = i.frames / i.samplerate
    else:
        durasi[f] = float(durasi_t1[f])
    assert abs(durasi[f] - durasi_t1[f]) < 0.01, f"{f}: durasi berbeda dengan Tahap 1"

print(f"Varian: {VARIAN}, baseline {KODE_DASAR}; batch Tahap 3 = {pemilihan['batch_tahap3']}")
print(f"{len(uji)} rekaman uji, {sum(durasi.values())/3600:.2f} jam; WAV ditemukan: {sum(audio_ada.values())}/{len(uji)}")
print(f"Kamus terkunci sejak {kamus.tanggal_kunci}")

Varian: large-v3, baseline T1-E; batch Tahap 3 = 1
9 rekaman uji, 2.44 jam; WAV ditemukan: 9/9
Kamus terkunci sejak 2026-10-13


In [6]:
def nvsmi(q):
    try:
        out = subprocess.check_output(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"],
                                      stderr=subprocess.DEVNULL)
        return out.decode(errors="replace").strip().splitlines()[0].strip()
    except Exception:
        return None

GPU = {"nama": nvsmi("name"), "vram_total_mb": nvsmi("memory.total"),
       "vram_terpakai_mb": nvsmi("memory.used"), "driver": nvsmi("driver_version")}
print(GPU)
if GPU["vram_terpakai_mb"] and int(GPU["vram_terpakai_mb"]) > 400:
    print("[!] VRAM terpakai sebelum mulai lebih dari 400 MB. Alihkan tampilan ke iGPU dan tutup aplikasi lain.")

{'nama': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU', 'vram_total_mb': '6144', 'vram_terpakai_mb': '0', 'driver': '577.02'}


## 1. Daftar hotwords per mata kuliah
Subbab 3.6.6 dan Gambar 4.7. Untuk setiap mata kuliah:
1. Setiap entri glosarium diwakili satu variasi sebutan, yaitu yang paling sering muncul di materi ajar. Bila seri, dipilih yang sama dengan bentuk kanonik, lalu menurut abjad.
2. Entri diurutkan dari yang paling sering muncul di materi ajar (seri menurut abjad), lalu dimasukkan satu per satu.
3. Setiap kali entri ditambahkan, daftar diurutkan menurut abjad, digabung dengan koma, dan jumlah tokennya dihitung persis seperti faster-whisper mengodekannya (`" " + hotwords`). Penambahan berhenti pada entri pertama yang membuat daftar melebihi 223 token.

Tokenizer dibaca dari berkas model yang sama dengan yang dipakai transkripsi, tanpa memuat model ke GPU.

In [7]:
from faster_whisper.utils import download_model
from tokenizers import Tokenizer

TOKEN_MAKS = 223
tokenizer = Tokenizer.from_file(str(Path(download_model(VARIAN)) / "tokenizer.json"))

def n_token(teks):
    # sama dengan get_prompt pada faster_whisper/transcribe.py: tokenizer.encode(" " + hotwords.strip())
    return len(tokenizer.encode(" " + teks.strip(), add_special_tokens=False).ids)

frek = pd.read_csv(FREKUENSI, keep_default_na=False, encoding="utf-8-sig")
frek["abjad"] = frek.bentuk_permukaan.str.casefold()
frek["kanonik"] = frek.abjad == frek.bentuk_kanonik.str.casefold()
sebutan = (frek.sort_values(["entri_id", "frekuensi_bentuk", "kanonik", "abjad"], ascending=[True, False, False, True])
               .drop_duplicates("entri_id"))

baris, HOTWORDS = [], {}
for mk, g in sebutan.groupby("mata_kuliah"):
    g = g.sort_values(["frekuensi_entri", "abjad"], ascending=[False, True])
    masuk, berhenti = [], False
    for urutan, r in enumerate(g.itertuples(), 1):
        n = None
        if not berhenti:
            calon = sorted(masuk + [r.bentuk_permukaan], key=str.casefold)
            n = n_token(", ".join(calon))
            berhenti = n > TOKEN_MAKS
            if not berhenti:
                masuk = calon
        baris.append({"mata_kuliah": mk, "urutan": urutan, "entri_id": r.entri_id, "bentuk_kanonik": r.bentuk_kanonik,
                      "sebutan": r.bentuk_permukaan, "frekuensi_sebutan": r.frekuensi_bentuk,
                      "frekuensi_entri": r.frekuensi_entri, "token_daftar": n,
                      "masuk": r.bentuk_permukaan in masuk})
    HOTWORDS[mk] = ", ".join(masuk)
    assert n_token(HOTWORDS[mk]) <= TOKEN_MAKS

daftar = pd.DataFrame(baris)
daftar.to_csv(T2 / "hotwords.csv", index=False, encoding="utf-8-sig")
(T2 / "hotwords.json").write_text(json.dumps(
    {mk: {"hotwords": h, "n_token": n_token(h), "n_entri": h.count(", ") + 1} for mk, h in HOTWORDS.items()},
    ensure_ascii=False, indent=2), encoding="utf-8")

r = daftar.groupby("mata_kuliah").agg(entri=("entri_id", "size"), masuk=("masuk", "sum"),
                                      frekuensi_terendah_masuk=("frekuensi_entri", lambda s: s[daftar.loc[s.index, "masuk"]].min()))
r["token"] = [n_token(HOTWORDS[mk]) for mk in r.index]
r["rekaman_uji"] = [sum(MK[f] == mk for f in uji) for mk in r.index]
print(r.to_string())
print()
for mk, h in HOTWORDS.items():
    print(f"{mk:7s} {h[:110]}...")

             entri  masuk  frekuensi_terendah_masuk  token  rekaman_uji
mata_kuliah                                                            
AOK            150     71                        14    221            2
APS            150     73                         4    223            2
IMK            150     89                         4    221            1
JARNIR         150     54                         8    218            1
PKK            150     62                        10    221            1
RPL            150     70                         5    221            2

AOK     ALU, aritmatika, ARM, arsitektur, biner, bit, branch, buffer, bus, byte, cache, chip, CISC, cloud computing, c...
APS     abstraction, adaptability, aggregation, aktor, aliran alternatif, aliran utama, ambigu, analisis, association,...
IMK     affordance, alignment, auditory, batch processing, buttons, colour, colour blindness, computer, consistency, c...
JARNIR  802.11, ack, agregasi, amplitudo, AP, bandwidth, 

## 2. Transkripsi T2
Model, presisi, parameter decoding, tiga ulangan, dan percobaan awal sama dengan Tahap 1. Hotwords setiap rekaman mengikuti mata kuliahnya, dan percobaan awal memakai daftar mata kuliah rekamannya. Seluruh rekaman dikerjakan dalam satu proses (`src/transkripsi.py`).

In [8]:
PARAMETER = meta_t1["parameter_decoding"]
COMPUTE_TYPE = meta_t1["compute_type"]
ULANGAN = meta_t1["rtf"]["ulangan"]
PEMANASAN = meta_t1["rtf"]["pemanasan"]
DEVICE = "cuda"
assert "hotwords" not in PARAMETER and PARAMETER["condition_on_previous_text"] is False

print(pd.Series({k: str(v) for k, v in PARAMETER.items()} | {"hotwords": "per mata kuliah (Bagian 1)",
                                                           "presisi": COMPUTE_TYPE}).to_string())
print(f"\nPercobaan awal: {PEMANASAN['detik']} detik pertama {PEMANASAN['file_id']}; ulangan terukur: {ULANGAN}")

language                                                  id
task                                              transcribe
beam_size                                                  5
temperature                   [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
vad_filter                                              True
condition_on_previous_text                             False
word_timestamps                                        False
hotwords                          per mata kuliah (Bagian 1)
presisi                                              float16

Percobaan awal: 120 detik pertama APS-05; ulangan terukur: 3


In [9]:
JALANKAN_TRANSKRIPSI = True

if JALANKAN_TRANSKRIPSI:
    assert all(audio_ada.values()), f"WAV tidak ditemukan: {[f for f, a in audio_ada.items() if not a]}"
    print(f"Audio yang diproses: {sum(durasi.values())/3600:.2f} jam x {ULANGAN} ulangan\n")
    print(f"VRAM terpakai sebelum mulai: {nvsmi('memory.used')} MB")
    fp = PEMANASAN["file_id"]
    cfg = {"nama": KODE, "model": VARIAN, "device": DEVICE, "compute_type": COMPUTE_TYPE,
           "parameter": PARAMETER, "ulangan": ULANGAN,
           "rekaman": [{"file_id": f, "path": str(AUDIO_PROC / f"{f}.wav"), "durasi": round(durasi[f], 3),
                        "parameter": {"hotwords": HOTWORDS[MK[f]]}} for f in uji],
           "pemanasan": {"path": str(AUDIO_PROC / f"{fp}.wav"), "detik": PEMANASAN["detik"],
                         "parameter": {"hotwords": HOTWORDS[MK[fp]]}},
           "arsip_mentah": str(MENTAH), "log_waktu": str(LOG_WAKTU)}
    tr.jalankan_terpisah(cfg, KERJA / "transkripsi_t2.json", cwd=ROOT)
else:
    print("JALANKAN_TRANSKRIPSI False: memakai hasil yang sudah ada di", rel(T2))

Audio yang diproses: 2.44 jam x 3 ulangan

VRAM terpakai sebelum mulai: 0 MB
[T2] 0 sudah ada, 27 dikerjakan
[T2] model dimuat dalam 6.0 detik
[T2] pemanasan 120 detik audio selesai dalam 16.9 detik
[T2] ulangan 1/3 AOK-04      183.7 detik  RTF 0.1567
[T2] ulangan 1/3 AOK-07      152.1 detik  RTF 0.1630
[T2] ulangan 1/3 APS-03       82.0 detik  RTF 0.1276
[T2] ulangan 1/3 APS-05      102.7 detik  RTF 0.1767
[T2] ulangan 1/3 IMK-02      214.1 detik  RTF 0.1907
[T2] ulangan 1/3 JARNIR-03   505.3 detik  RTF 0.4636
[T2] ulangan 1/3 PKK-02      118.0 detik  RTF 0.1166
[T2] ulangan 1/3 RPL-07      236.5 detik  RTF 0.1948
[T2] ulangan 1/3 RPL-10      200.3 detik  RTF 0.1987
[T2] ulangan 2/3 AOK-04      174.9 detik  RTF 0.1492
[T2] ulangan 2/3 AOK-07      183.9 detik  RTF 0.1970
[T2] ulangan 2/3 APS-03       83.3 detik  RTF 0.1297
[T2] ulangan 2/3 APS-05      109.5 detik  RTF 0.1882
[T2] ulangan 2/3 IMK-02      214.9 detik  RTF 0.1915
[T2] ulangan 2/3 JARNIR-03   478.4 detik  RTF 0.4389
[T2] u

In [10]:
assert LOG_WAKTU.exists(), "Belum ada hasil transkripsi T2. Jalankan sel sebelumnya dengan JALANKAN_TRANSKRIPSI = True."
waktu = pd.DataFrame([json.loads(b) for b in LOG_WAKTU.read_text(encoding="utf-8").splitlines() if b.strip()])
waktu = waktu[(waktu.konfigurasi == KODE) & waktu.file_id.isin(uji)]
assert len(waktu) == ULANGAN * len(uji), f"baru {len(waktu)} dari {ULANGAN * len(uji)} pasangan ulangan x rekaman"

# Keluaran mentah harus lengkap dan memakai hotwords mata kuliahnya
for f in uji:
    p = MENTAH / f"{f}.json"
    assert p.exists(), f"keluaran mentah {f} tidak ada"
    assert json.loads(p.read_text(encoding="utf-8"))["parameter"].get("hotwords") == HOTWORDS[MK[f]], \
        f"{f}: hotwords di keluaran mentah berbeda dengan Bagian 1"

sidik = waktu.pivot_table(index="file_id", columns="ulangan", values="sidik", aggfunc="first")
beda = sidik[sidik.nunique(axis=1) > 1]
print(f"{len(waktu)} pasangan ulangan x rekaman tercatat; keluaran berbeda antarulangan: {len(beda)} dari {len(sidik)} rekaman")
if len(beda):
    print(beda.to_string())

27 pasangan ulangan x rekaman tercatat; keluaran berbeda antarulangan: 4 dari 9 rekaman
ulangan                   1                 2                 3
file_id                                                        
AOK-04     51d53db437c80e97  192c83cef1220a12  b8fc4df1d76ef382
AOK-07     958233db2a4eb75c  39648a7ab40043bd  922e9e4b514ace82
APS-05     6adb5af25685c189  f87992eab876bf1e  d115a76395cfd602
JARNIR-03  dd5992ec169ba69f  a17b4fdba44fb71c  935fbf4dada0ac64


## 3. Metrik dan log per data
Prosedur subbab 3.6.8, sama dengan Tahap 1: segmen di luar acuan dibuang, hipotesis dinormalisasi, satu penyelarasan per rekaman, lalu dibagi ke blok tiga menit. `src/log_data.py` menghitung metrik dan sekaligus menulis log per rekaman, segmen, istilah, dan kata ke `tahap2/log/`, dengan format yang sama dengan `tahap1/log/`.

In [11]:
hasil = ld.log_konfigurasi(KODE, KODE, MENTAH, acuan, norm, durasi, manifest, waktu, kanonik)
blok = hasil["blok"]
blok.to_csv(T2 / "metrik_blok.csv", index=False, encoding="utf-8-sig")
hasil["verifikasi"].to_csv(T2 / "verifikasi_jiwer.csv", index=False, encoding="utf-8-sig")
assert hasil["verifikasi"].sama.all(), "WER penyelarasan berbeda dengan jiwer"
assert len(blok) == len(pd.read_csv(T1 / "metrik_blok.csv").query("kode == @KODE_DASAR")), "jumlah blok berbeda dengan baseline"

jumlah = ld.tulis_log([hasil], T2 / "log")
print(f"WER sama dengan jiwer: {hasil['verifikasi'].sama.sum()} dari {len(uji)} rekaman; {len(blok)} blok")
print("Log ditulis ke", rel(T2 / "log"), {k: v for k, v in jumlah.items()})

rek = hasil["rekaman"]
t = rek.set_index("file_id")[["mata_kuliah", "jalur_acuan", "segmen", "segmen_dibuang", "istilah_salah",
                              "istilah_muncul", "WER", "CER", "TTER", "rtf_rerata"]].copy()
t[["WER", "CER", "TTER"]] = (100 * t[["WER", "CER", "TTER"]]).round(2)
t["rtf_rerata"] = t.rtf_rerata.round(4)
print("\nPer rekaman (WER, CER, TTER dalam persen):")
print(t.to_string())

WER sama dengan jiwer: 9 dari 9 rekaman; 54 blok
Log ditulis ke data/output/tahap2/log {'rekaman.csv': 9, 'segmen_hipotesis.csv': 1223, 'kemunculan_istilah.csv': 1149, 'penyelarasan/*.tsv': 9}

Per rekaman (WER, CER, TTER dalam persen):
          mata_kuliah jalur_acuan  segmen  segmen_dibuang  istilah_salah  istilah_muncul    WER    CER   TTER  rtf_rerata
file_id                                                                                                                  
AOK-04            AOK     whisper     124               0             38             234  10.12   5.96  15.03      0.1525
AOK-07            AOK     whisper     110               0             10             130   7.84   4.26   6.55      0.1769
APS-03            APS     whisper      67               0              8              94   5.01   3.18   8.00      0.1292
APS-05            APS     whisper      50               0             12             144   8.66   4.85   6.81      0.1660
IMK-02            IMK     whisp

In [12]:
# Agregat T2 dan RTF, dengan kolom yang sama seperti tahap1/metrik_varian.csv
rtf_u = waktu.groupby("ulangan").agg(proses=("proses_s", "sum"), durasi=("durasi_s", "sum"))
rtf_u["rtf"] = rtf_u.proses / rtf_u.durasi
rtf_u.reset_index().assign(konfigurasi=KODE).to_csv(T2 / "rtf_ulangan.csv", index=False, encoding="utf-8-sig")
rtf_rek = waktu.groupby("file_id").rtf.mean()
sesi = waktu.sort_values("waktu").iloc[0]

agregat = an.gabung(blok, "kode")
agregat.insert(0, "varian", VARIAN)
agregat = agregat.assign(RTF=rtf_u.rtf.mean(), RTF_sd=rtf_u.rtf.std(ddof=1),
                         RTF_min_rekaman=rtf_rek.min(), RTF_maks_rekaman=rtf_rek.max(),
                         muat_model_s=sesi.muat_s, vram_puncak_mb=sesi.vram_puncak_mb - sesi.vram_awal_mb)
agregat.to_csv(T2 / "metrik_t2.csv", encoding="utf-8-sig")

dasar = pd.read_csv(T1 / "metrik_varian.csv", encoding="utf-8-sig").set_index("kode").loc[[KODE_DASAR]]
banding = pd.concat([dasar, agregat])[["WER", "CER", "TTER", "RTF", "RTF_sd", "vram_puncak_mb"]]
banding.loc["selisih"] = banding.loc[KODE] - banding.loc[KODE_DASAR]
banding[["WER", "CER", "TTER"]] = (100 * banding[["WER", "CER", "TTER"]]).round(2)
banding[["RTF", "RTF_sd"]] = banding[["RTF", "RTF_sd"]].round(4)
print("Agregat (WER, CER, TTER dalam persen; selisih dalam poin persentase):")
print(banding.to_string())

Agregat (WER, CER, TTER dalam persen; selisih dalam poin persentase):
          WER   CER   TTER     RTF  RTF_sd  vram_puncak_mb
kode                                                      
T1-E     4.63  1.70   5.05  0.1619  0.0098          3649.0
T2       9.03  4.66  10.46  0.1993  0.0094          4033.0
selisih  4.40  2.96   5.41  0.0373 -0.0003           384.0


## 4. Analisis terhadap baseline
Subbab 3.4. Baseline adalah baris Tahap 1 untuk varian terpilih, dibaca dari `tahap1/` tanpa dihitung ulang. Seluruh perbandingan berpasangan atas rekaman dan blok yang sama.

In [13]:
blok_dasar = (pd.read_csv(T1 / "metrik_blok.csv", encoding="utf-8-sig").query("kode == @KODE_DASAR")
                .rename(columns={"varian": "konfigurasi"}))
# metrik dihitung ulang dari bilangan bulat supaya pembulatan CSV tidak terbaca sebagai selisih
blok_gab = an.metrik_dari_jumlah(pd.concat([blok_dasar, blok], ignore_index=True))
rek_dasar = pd.read_csv(T1 / "log" / "rekaman.csv", encoding="utf-8-sig").query("kode == @KODE_DASAR")
rek_gab = an.metrik_dari_jumlah(pd.concat([rek_dasar, rek], ignore_index=True))
kem_dasar = (pd.read_csv(T1 / "log" / "kemunculan_istilah.csv", keep_default_na=False, encoding="utf-8-sig")
               .query("konfigurasi == @VARIAN"))
URUT = [KODE_DASAR, KODE]
print(f"Baseline {KODE_DASAR}: {len(blok_dasar)} blok, {len(rek_dasar)} rekaman, {len(kem_dasar)} kemunculan istilah")

Baseline T1-E: 54 blok, 9 rekaman, 1149 kemunculan istilah


### 4.1 Statistik deskriptif dan ambang
Format: agregat (rerata blok ± sd; min-maks). WER dibandingkan dengan ambang 25 persen (Munteanu, et al., 2006) dan RTF dengan ambang 1 (Pratap, et al., 2020).

In [14]:
desk = an.deskriptif(blok_gab, "kode").loc[URUT]
rtf = pd.DataFrame({"RTF_rerata": banding.loc[URUT, "RTF"], "RTF_sd": banding.loc[URUT, "RTF_sd"]})
desk = desk.join(rtf)
desk.round(4).to_csv(T2 / "statistik_deskriptif.csv", encoding="utf-8-sig")

ringkas = lambda m: desk.apply(lambda r: f"{r[f'{m}_agregat']:.2f} ({r[f'{m}_rerata']:.2f} ± {r[f'{m}_sd']:.2f}; "
                                          f"{r[f'{m}_min']:.2f}-{r[f'{m}_maks']:.2f})", axis=1)
print(pd.DataFrame({f"{m} %": ringkas(m) for m in an.METRIK}
                   | {"RTF": desk.apply(lambda r: f"{r.RTF_rerata:.4f} ± {r.RTF_sd:.4f}", axis=1)}).to_string())

ambang = pd.DataFrame({
    "WER %": desk.WER_agregat.round(2),
    "WER <= 25%": desk.WER_agregat <= 25,
    "blok dengan WER <= 25%": blok_gab.groupby("kode").WER.apply(lambda s: f"{(s <= 0.25).sum()}/{s.notna().sum()}").loc[URUT],
    "RTF": desk.RTF_rerata.round(4),
    "RTF <= 1": desk.RTF_rerata <= 1,
})
ambang.to_csv(T2 / "ambang.csv", encoding="utf-8-sig")
print()
print(ambang.to_string())

                                 WER %                           CER %                              TTER %              RTF
kode                                                                                                                       
T1-E    4.63 (5.49 ± 7.17; 0.00-33.33)  1.70 (2.28 ± 3.99; 0.00-21.59)    5.05 (7.23 ± 16.68; 0.00-100.00)  0.1619 ± 0.0098
T2    9.03 (10.50 ± 11.65; 0.00-61.11)  4.66 (6.04 ± 8.83; 0.00-46.00)  10.46 (12.36 ± 17.62; 0.00-100.00)  0.1993 ± 0.0094

      WER %  WER <= 25% blok dengan WER <= 25%     RTF  RTF <= 1
kode                                                            
T1-E   4.63        True                  52/54  0.1619      True
T2     9.03        True                  50/54  0.1993      True


### 4.2 Uji Wilcoxon T2 terhadap baseline
Tingkat blok (n = 54, TTER 53) dengan ambang p < 0,001 sebagai dasar kesimpulan, dan tingkat konfigurasi atas nilai tiap rekaman (n = 9) sebagai pelengkap. `selisih_median` adalah T2 dikurangi baseline dalam poin persentase; nilai negatif berarti T2 lebih rendah kesalahannya.

In [15]:
ALFA_BLOK = meta_t1["metrik"]["ambang_p_blok"]
baris = []
for tingkat, data, idx in (("blok", blok_gab, ["file_id", "blok"]), ("konfigurasi", rek_gab, ["file_id"])):
    for m in an.METRIK:
        tab = data.pivot_table(index=idx, columns="kode", values=m).dropna()
        baris.append({"tingkat": tingkat, "metrik": m, "konfigurasi_A": KODE, "konfigurasi_B": KODE_DASAR,
                      **an.uji_wilcoxon(tab[KODE], tab[KODE_DASAR])})
wil = pd.DataFrame(baris)
wil["signifikan"] = np.where(wil.tingkat == "blok", np.where(wil.p < ALFA_BLOK, "ya", "tidak"), "-")
wil.to_csv(T2 / "wilcoxon.csv", index=False, encoding="utf-8-sig")

t = wil.pivot_table(index="metrik", columns="tingkat", values="p", sort=False)
b = wil[wil.tingkat == "blok"].set_index("metrik")
t = t.assign(selisih_median=b.selisih_median.round(2), n_tak_nol=b.n_tak_nol, signifikan=b.signifikan)
print(t[["selisih_median", "n_tak_nol", "blok", "konfigurasi", "signifikan"]]
      .to_string(formatters={"blok": "{:.2e}".format, "konfigurasi": "{:.4f}".format}))

tingkat  selisih_median  n_tak_nol     blok konfigurasi signifikan
metrik                                                            
WER                1.98         43 2.69e-07      0.0117         ya
CER                1.22         46 4.35e-07      0.0195         ya
TTER               0.00         30 2.41e-03      0.0156      tidak


### 4.3 Perubahan WER dan TTER per rekaman
Disajikan berdampingan (Ugan, et al., 2025) supaya terlihat rekaman yang kedua metriknya bergerak berlawanan arah. Selisih dalam poin persentase.

In [16]:
p = rek_gab.pivot_table(index="file_id", columns="kode", values=["WER", "TTER"])
ubah = pd.DataFrame({
    "mata_kuliah": rek.set_index("file_id").mata_kuliah, "jalur_acuan": rek.set_index("file_id").jalur_acuan,
    f"WER_{KODE_DASAR}": 100 * p["WER"][KODE_DASAR], f"WER_{KODE}": 100 * p["WER"][KODE],
    "selisih_WER": 100 * (p["WER"][KODE] - p["WER"][KODE_DASAR]),
    f"TTER_{KODE_DASAR}": 100 * p["TTER"][KODE_DASAR], f"TTER_{KODE}": 100 * p["TTER"][KODE],
    "selisih_TTER": 100 * (p["TTER"][KODE] - p["TTER"][KODE_DASAR]),
}).loc[uji]
ubah["arah"] = np.where(np.sign(ubah.selisih_WER) * np.sign(ubah.selisih_TTER) < 0, "berlawanan", "")
ubah.round(2).to_csv(T2 / "perubahan_rekaman.csv", encoding="utf-8-sig")
print(ubah.round(2).to_string())

          mata_kuliah jalur_acuan  WER_T1-E  WER_T2  selisih_WER  TTER_T1-E  TTER_T2  selisih_TTER arah
file_id                                                                                                
AOK-04            AOK     whisper      2.27   10.12         7.85       8.39    15.03          6.64     
AOK-07            AOK     whisper      1.10    7.84         6.74       4.17     6.55          2.38     
APS-03            APS     whisper      0.26    5.01         4.75       0.80     8.00          7.20     
APS-05            APS     whisper      0.38    8.66         8.28       1.05     6.81          5.76     
IMK-02            IMK     whisper      0.00    0.35         0.35       0.00     2.80          2.80     
JARNIR-03      JARNIR    wav2vec2     10.79   25.59        14.79       1.85    26.85         25.00     
PKK-02            PKK    wav2vec2     16.70   20.61         3.91      16.36    20.00          3.64     
RPL-07            RPL    wav2vec2      5.79    5.61        -0.18

### 4.4 Rincian per jalur acuan
Acuan jalur whisper disusun dari keluaran large-v3 lalu dikoreksi, sehingga baseline di jalur itu mendekati batas atas dan setiap perubahan keluaran cenderung terhitung sebagai kesalahan baru. Tabel ini deskriptif, hanya untuk membantu menafsirkan Bagian 4.1 dan 4.2.

In [17]:
j = an.gabung(rek_gab, ["jalur_acuan", "kode"])[an.METRIK].mul(100).unstack("kode")
for m in an.METRIK:
    j[(m, "selisih")] = j[(m, KODE)] - j[(m, KODE_DASAR)]
j = j[[(m, k) for m in an.METRIK for k in URUT + ["selisih"]]]
j["rekaman"] = rek.groupby("jalur_acuan").size()
j.round(2).to_csv(T2 / "per_jalur.csv", encoding="utf-8-sig")
print(j.round(2).to_string())

              WER                  CER                TTER                rekaman
kode         T1-E     T2 selisih  T1-E    T2 selisih  T1-E     T2 selisih        
jalur_acuan                                                                      
whisper      0.82   5.80    4.99  0.24  3.21    2.97  3.88   9.12    5.25       5
wav2vec2     8.36  12.19    3.83  3.09  6.04    2.95  6.99  12.67    5.67       4


### 4.5 Perubahan per istilah
Setiap kemunculan istilah pada acuan dipasangkan antara baseline dan T2, lalu digolongkan menjadi *tetap benar*, *diperbaiki*, *dirusak*, atau *tetap salah*. Kolom `di_hotwords` menandai entri yang termuat dalam daftar mata kuliahnya.

In [18]:
entri_masuk = set(daftar.loc[daftar.masuk, "entri_id"])
pi = an.bandingkan_istilah(kem_dasar, hasil["istilah"])
pi["di_hotwords"] = pi.entri_id.isin(entri_masuk)
pi.to_csv(T2 / "perubahan_istilah.csv", index=False, encoding="utf-8-sig")

print("Kemunculan istilah menurut status:")
print(pd.crosstab(pi.di_hotwords, pi.status, margins=True, margins_name="total").to_string())

per_entri = (pi.groupby(["mata_kuliah", "entri_id", "bentuk_kanonik", "di_hotwords"]).status
               .value_counts().unstack(fill_value=0).reset_index())
for s in ("diperbaiki", "dirusak", "tetap salah", "tetap benar"):
    if s not in per_entri:
        per_entri[s] = 0
per_entri.to_csv(T2 / "perubahan_istilah_entri.csv", index=False, encoding="utf-8-sig")
kolom = ["mata_kuliah", "bentuk_kanonik", "di_hotwords", "diperbaiki", "dirusak", "tetap salah"]
for s in ("diperbaiki", "dirusak"):
    t = per_entri[per_entri[s] > 0].sort_values(s, ascending=False).head(10)
    print(f"\nEntri yang paling banyak {s}:")
    print(t[kolom].to_string(index=False) if len(t) else "  tidak ada")

Kemunculan istilah menurut status:
status       diperbaiki  dirusak  tetap benar  tetap salah  total
di_hotwords                                                      
False                 1       19          166            4    190
True                 24       71          833           31    959
total                25       90          999           35   1149

Entri yang paling banyak diperbaiki:
mata_kuliah           bentuk_kanonik  di_hotwords  diperbaiki  dirusak  tetap salah
        AOK                    cache         True          13        3            7
        AOK                      I/O         True           2        2            1
        AOK               I/O module         True           2        1            1
        AOK              main memory         True           2        1            0
        PKK                brainware         True           2        0            0
        AOK                   memori         True           1       18            0
        A

### 4.6 Ekspor kesalahan istilah dan kandidat halusinasi untuk dilabeli
Subbab 3.4 butir 5, dengan aturan yang sama seperti Tahap 1, diturunkan dari log Bagian 3. Kolom `golongan` diisi `substitusi_mirip`, `penghapusan`, `transliterasi`, atau `lainnya`; kolom `halusinasi` diisi `ya` atau `tidak`. Berkas yang sudah mulai dilabeli tidak ditimpa.

In [19]:
def sudah_dilabeli(p, kolom):
    if not p.exists():
        return False
    lama = pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    return bool((lama[kolom].ne("") & lama[kolom].ne("penghapusan")).any() or lama.catatan.ne("").any())

for berkas, df, kolom in (("kesalahan_istilah.csv", an.kesalahan_istilah(hasil["istilah"]), "golongan"),
                          ("kandidat_halusinasi.csv", an.kandidat_halusinasi(T2 / "log", KODE), "halusinasi")):
    p = T2 / berkas
    if sudah_dilabeli(p, kolom):
        print(f"{berkas}: sudah berisi label, tidak ditimpa")
        continue
    df.to_csv(p, index=False, encoding="utf-8-sig")
    rinci = (df.sumber.value_counts().to_dict() if "sumber" in df
             else {"terisi otomatis": int((df.golongan != "").sum()), "perlu dilabeli": int((df.golongan == "").sum())})
    print(f"{berkas}: {len(df)} baris {rinci}")

kesalahan_istilah.csv: 125 baris {'terisi otomatis': 31, 'perlu dilabeli': 94}
kandidat_halusinasi.csv: 18 baris {'penyisipan beruntun': 9, 'segmen di luar acuan': 9}


## 5. Metadata run

In [20]:
import platform, importlib.metadata as im
def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return None

meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "Tahap 2: contextual biasing melalui hotwords",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": GPU,
    "varian": VARIAN, "kode": KODE, "kode_baseline": KODE_DASAR,
    "parameter_decoding": PARAMETER, "compute_type": COMPUTE_TYPE,
    "sumber_parameter": rel(T1 / "run_meta_tahap1.json"),
    "hotwords": {"berkas": rel(T2 / "hotwords.json"), "token_maks": TOKEN_MAKS,
                 "per_mata_kuliah": {mk: {"n_entri": h.count(", ") + 1, "n_token": n_token(h)} for mk, h in HOTWORDS.items()}},
    "rtf": {"ulangan": ULANGAN, "pemanasan": PEMANASAN},
    "metrik": {"panjang_blok_detik": mt.PANJANG_BLOK, "ambang_p_blok": ALFA_BLOK},
    "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
    "versi_pustaka": {p: versi(p) for p in ["faster-whisper", "ctranslate2", "onnxruntime", "tokenizers", "jiwer",
                                             "scipy", "numpy", "pandas", "transformers", "soundfile"]},
}
(T2 / "run_meta_tahap2.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

{
  "faster-whisper": "1.2.1",
  "ctranslate2": "4.7.2",
  "onnxruntime": "1.26.0",
  "tokenizers": "0.20.0",
  "jiwer": "4.0.0",
  "scipy": "1.17.1",
  "numpy": "2.0.2",
  "pandas": "3.0.3",
  "transformers": "4.45.2",
  "soundfile": "0.13.1"
}


## Output

| Berkas di `data/output/tahap2/` | Isi | Dipakai di |
|---|---|---|
| `hotwords.csv`, `hotwords.json` | Urutan entri, sebutan terpilih, jumlah token, entri yang masuk, dan teks hotwords per mata kuliah | Bab 6, lampiran |
| `mentah/<ID>.json` | Keluaran mentah ulangan pertama, tidak disunting, beserta hotwords yang dipakai | Pemeriksaan ulang, lampiran |
| `waktu_transkripsi.jsonl`, `rtf_ulangan.csv` | Waktu proses tiap ulangan, RTF, waktu muat, puncak VRAM | Bab 6, analisis sumber daya |
| `metrik_blok.csv`, `metrik_t2.csv` | Pembilang, penyebut, WER, CER, TTER per blok dan agregat T2 beserta RTF | Bab 6 |
| `verifikasi_jiwer.csv` | Bukti penyelarasan | Lampiran |
| `log/rekaman.csv` | Metrik tiap rekaman beserta mata kuliah, jalur acuan, durasi, dan RTF tiap ulangan | Penelusuran rekaman dengan metrik buruk |
| `log/segmen_hipotesis.csv` | Tiap segmen keluaran model: teks, waktu, dipakai atau dibuang, info decoding | Penelusuran sebab kesalahan dan halusinasi |
| `log/kemunculan_istilah.csv` | Tiap kemunculan istilah glosarium: benar atau salah, hipotesis, jenis operasi | Analisis kesalahan istilah |
| `log/penyelarasan/T2/<ID>.tsv` | Tiap kata hasil penyelarasan: kata acuan, kata hipotesis, operasi, waktu, info decoding | Penelusuran kesalahan per kata, lampiran |
| `statistik_deskriptif.csv`, `ambang.csv`, `wilcoxon.csv` | T2 terhadap baseline T1-E | Bab 6, rumusan masalah kedua |
| `perubahan_rekaman.csv`, `per_jalur.csv` | Perubahan WER dan TTER per rekaman dan per jalur acuan | Bab 6 |
| `perubahan_istilah.csv`, `perubahan_istilah_entri.csv` | Istilah yang diperbaiki dan dirusak hotwords, per kemunculan dan per entri | Bab 6, analisis kesalahan istilah |
| `kesalahan_istilah.csv`, `kandidat_halusinasi.csv` | Tabel untuk dilabeli manual | Analisis kesalahan istilah (3.4 butir 5) |